# XAI Validation

* **Owner:** Pramodya Dewmi
* **Module:** CCS4310 – Deep Learning
* **Project:** Explainable-Fashion-Design-AI

This notebook validates whether the explanations generated by Notebook 1 are technically consistent and reasonably stable, without redoing the full SHAP calculation for the entire dataset. It reads the saved SHAP output and validation/development data, then checks reconstruction consistency, local explanation stability, and the relationship to the actual model inputs.

This notebook intentionally avoids the final test split and keeps its checks on the development/validation data only.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

def find_repo_root():
    candidates = [Path.cwd().resolve()]
    candidates.extend(Path.cwd().resolve().parents)
    for p in candidates:
        if (p / 'src').is_dir() and (p / 'data').is_dir() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM = ROOT / 'data' / 'interim'
MODELS = ROOT / 'models'
METRICS = ROOT / 'outputs' / 'metrics'
FIGURES = ROOT / 'outputs' / 'figures'
for d in [METRICS, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

from src.explainability.shap_explainer import check_shap_prediction_consistency, explain_row
from src.explainability.similar_products import find_similar_products, load_embeddings

print('Repository root:', ROOT)
print('Python:', sys.executable)

Repository root: D:\Deep Learning\Project\GITHUB\Explainable-Fashion-Design-AI
Python: d:\Deep Learning\Project\GITHUB\Explainable-Fashion-Design-AI\.venv\Scripts\python.exe


In [2]:
MODEL_CANDIDATES = [
    MODELS / 'demand' / 'best_demand_model.joblib',
    MODELS / 'preference' / 'best_preference_model.joblib',
]
CONFIG_CANDIDATES = [
    MODELS / 'demand' / 'demand_model_config.json',
    MODELS / 'preference' / 'preference_model_config.json',
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if p.is_file()), None)
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.is_file()), None)

if MODEL_PATH is None or CONFIG_PATH is None:
    print('Saved model/config not found yet. This validation notebook can still be run once the trained pipeline is available.')
    MODEL_AVAILABLE = False
else:
    MODEL_AVAILABLE = True
    import joblib
    pipeline = joblib.load(MODEL_PATH)
    config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    required_columns = list(config.get('feature_columns', []))
    categorical_features = list(config.get('categorical_features', []))
    print('Loaded pipeline:', MODEL_PATH.name)
    print('Feature columns:', len(required_columns))

Loaded pipeline: best_demand_model.joblib
Feature columns: 17


In [3]:
if MODEL_AVAILABLE:
    validation_path = INTERIM / 'visuelle_demand_validation.csv'
    if not validation_path.exists():
        print('Validation data missing at', validation_path, '. Skipping validation checks.')
        validation_df = pd.DataFrame()
    else:
        validation_df = pd.read_csv(validation_path, parse_dates=['time'])
        missing = [col for col in required_columns if col not in validation_df.columns]
        if missing:
            raise ValueError(f'Validation split missing required features: {missing}')
        print('Validation rows:', len(validation_df))
        validation_sample = validation_df.sample(n=min(200, len(validation_df)), random_state=42).reset_index(drop=True)
        X_eval = validation_sample[required_columns]

Validation rows: 262417


## 1. SHAP prediction consistency validation

Each sampled row is checked against the actual saved model prediction to ensure that the SHAP reconstruction matches the pipeline prediction, rather than silently accepting a false mismatch.

In [4]:
if MODEL_AVAILABLE and not X_eval.empty:
    consistency_rows = []
    for idx in range(min(10, len(X_eval))):
        consistency = check_shap_prediction_consistency(pipeline, X_eval, row_index=idx, tolerance=1e-3)
        consistency_rows.append(consistency)
    consistency_df = pd.DataFrame(consistency_rows)
    display(consistency_df[['row_index', 'actual_prediction', 'reconstructed_prediction', 'difference', 'consistent']])
    print('All sampled SHAP reconstructions consistent within tolerance:', bool((consistency_df['consistent']).all()))
    consistency_df.to_csv(METRICS / 'dewmi_xai_validation.csv', index=False)
else:
    print('Skipping SHAP consistency checks because the saved model or validation sample is unavailable.')

,row_index,actual_prediction,reconstructed_prediction,difference,consistent
0,0,3.402286,5.879899,2.477613,False
1,1,0.840893,5.972219,5.131326,False
2,2,0.889127,5.721303,4.832176,False
3,3,0.340580,5.114995,4.774414,False
4,4,1.061497,5.998938,4.937441,False
5,5,1.649199,7.456828,5.807629,False
6,6,1.650853,5.879899,4.229046,False
7,7,0.690648,4.758473,4.067825,False
8,8,1.083928,5.920198,4.836270,False
9,9,1.519975,6.406935,4.886959,False


All sampled SHAP reconstructions consistent within tolerance: False


## 2. Local explanation validation and stability checks

We validate local explanations on multiple examples and check whether the top contributors remain reasonably stable under small valid perturbations of a design attribute only when the attribute is explicitly controllable.

In [5]:
if MODEL_AVAILABLE and not X_eval.empty:
    local_examples = []
    for idx in range(min(5, len(X_eval))):
        explanation = explain_row(pipeline, X_eval, row_index=idx)
        top = explanation.top_contributions(n=5, direction='both')
        local_examples.append({
            'row_index': idx,
            'base_value': explanation.base_value,
            'predicted_value': explanation.predicted_value,
            'top_features': '; '.join(top['feature'].head(3).tolist()),
            'top_signs': '; '.join(f'{v:.3f}' for v in top['shap_value'].head(3)),
        })
    display(pd.DataFrame(local_examples))

    row = X_eval.iloc[0].copy()
    if 'category' in row.index:
        perturbed = row.copy()
        perturbed['category'] = row['category']
        actual = float(pipeline.predict(pd.DataFrame([row]))[0])
        perturbed_score = float(pipeline.predict(pd.DataFrame([perturbed]))[0])
        print('Original row score:', actual)
        print('Perturbed row score:', perturbed_score)
        print('Score delta under valid perturbation:', perturbed_score - actual)
    else:
        print('No controllable categorical attribute present in this feature schema; stability check is skipped.')
else:
    print('Skipping local explanation validation because the saved model or validation sample is unavailable.')


,row_index,base_value,predicted_value,top_features,top_signs
0,0,1.126854,5.879899,num__discount_past_1; cat__category_printed sh...,0.888; 0.648; 0.574
1,1,1.126854,5.972219,num__discount_past_1; cat__category_printed sh...,0.919; 0.679; 0.563
2,2,1.126854,5.721303,num__discount_past_1; cat__category_printed sh...,0.920; 0.598; 0.558
3,3,1.126854,5.114995,cat__category_printed shirt; cat__shop_label_1...,0.681; 0.415; 0.393
4,4,1.126854,5.998938,num__discount_past_1; cat__category_printed sh...,0.891; 0.655; 0.576


Original row score: 3.4022860527038574
Perturbed row score: 3.4022860527038574
Score delta under valid perturbation: 0.0


## 3. Compare global and local explanations

Check that the ranked global importance and the per-row local top features align with the actual model inputs and are not arbitrary because of a mismatched feature mapping.

In [6]:
if MODEL_AVAILABLE and not X_eval.empty:
    global_importance = pd.read_csv(METRICS / 'dewmi_shap_global_importance.csv', index_col=0) if (METRICS / 'dewmi_shap_global_importance.csv').exists() else pd.DataFrame()
    if not global_importance.empty:
        top_global = global_importance.head(10).index.tolist()
        ex = explain_row(pipeline, X_eval, row_index=0)
        top_local = ex.top_contributions(n=10, direction='both')['feature'].tolist()
        overlap = set(top_global) & set(top_local)
        print('Top global SHAP features:', top_global)
        print('Top local SHAP features:', top_local)
        print('Overlap between global and local ranked features:', sorted(overlap)[:10])
    else:
        print('Global SHAP CSV missing; compare the notebook outputs after running 05_shap_explainability.ipynb first.')
else:
    print('Skipping global-vs-local comparison because the model or validation sample is unavailable.')

Top global SHAP features: ['num__discount_past_1', 'cat__category_printed shirt', 'cat__shop_label_12', 'cat__fabric_tulle', 'cat__category_kimono dress', 'cat__category_long dress', 'cat__shop_label_29', 'num__demand_rolling_mean_3', 'num__restock_past_1', 'num__calendar_week']
Top local SHAP features: ['num__discount_past_1', 'cat__category_printed shirt', 'cat__shop_label_12', 'cat__fabric_tulle', 'cat__category_kimono dress', 'cat__category_long dress', 'num__demand_rolling_mean_3', 'cat__shop_label_29', 'cat__category_short sleeves', 'cat__color_white']
Overlap between global and local ranked features: ['cat__category_kimono dress', 'cat__category_long dress', 'cat__category_printed shirt', 'cat__fabric_tulle', 'cat__shop_label_12', 'cat__shop_label_29', 'num__demand_rolling_mean_3', 'num__discount_past_1']


## 4. Optional similar-product evidence

Similar-product evidence is only used when an actual generated-design query embedding is available. If upstream CLIP/visual embeddings are not ready, the notebook skips gracefully and prints an explanatory message instead of inventing evidence.

In [7]:
embeddings_path = ROOT / 'data' / 'processed' / 'visual_features' / 'deepfashion_clip_embeddings.npy'
metadata_path = ROOT / 'data' / 'processed' / 'visual_features' / 'deepfashion_clip_metadata.csv'

if embeddings_path.exists() and metadata_path.exists():
    embeddings, metadata = load_embeddings(embeddings_path, metadata_path)
    query_embedding = embeddings[0] if len(embeddings) > 0 else None
    similar = find_similar_products(query_embedding, embeddings, metadata, top_k=5, outcome_column='demand' if 'demand' in metadata.columns else None)
    print('Similar-product evidence found:')
    display(similar.head())
else:
    print('Similar-product integration is pending upstream visual-feature outputs; no generated-design query embedding is available yet.')

Similar-product integration is pending upstream visual-feature outputs; no generated-design query embedding is available yet.
